In [4]:
import import_ipynb
from Recodificacion import *
from Codigos import *

In [3]:
import pandas as pd

In [5]:
def features_histograma(path, prefix):
    import numpy as np

    f = {}

    hist, hist_norm = obtener_histograma_intensidades(path)

    # guardar todo
    f[f"{prefix}_hist"] = hist.tolist()
    f[f"{prefix}_hist_norm"] = hist_norm.tolist()
    f[f"{prefix}_hist_bins"] = list(range(256))

    # derivados
    hist_norm_safe = hist_norm[hist_norm > 0]

    f[f"{prefix}_hist_entropy"] = -np.sum(hist_norm_safe * np.log2(hist_norm_safe))
    f[f"{prefix}_hist_std"] = np.std(hist_norm)
    f[f"{prefix}_hist_mean"] = np.mean(hist_norm)
    f[f"{prefix}_hist_min"] = np.min(hist_norm)
    f[f"{prefix}_hist_max"] = np.max(hist_norm)

    mu = np.mean(hist_norm)
    f[f"{prefix}_hist_skew"] = np.mean((hist_norm - mu)**3)
    f[f"{prefix}_hist_kurt"] = np.mean((hist_norm - mu)**4)

    return f

In [6]:
def features_generales(path, prefix):
    f = {}

    # básicas
    f[f"{prefix}_rango"] = obtener_rango_dinamico(path)
    f[f"{prefix}_std"] = obtener_desv_estandar_brillo(path)

    # contraste
    f[f"{prefix}_michelson"] = obtener_contraste_michelson(path)

    # frecuencia
    f[f"{prefix}_hf"] = obtener_energia_altas_frecuencias(path)
    f[f"{prefix}_lf"] = obtener_energia_bajas_frecuencias(path)
    f[f"{prefix}_hlfr"] = obtener_hlfr(path)

    # bordes multi escala
    f[f"{prefix}_edges_s1"] = obtener_densidad_bordes(path, sigma=1.0)
    f[f"{prefix}_edges_s2"] = obtener_densidad_bordes(path, sigma=2.0)
    f[f"{prefix}_edges_s3"] = obtener_densidad_bordes(path, sigma=3.0)

    # gradiente
    f[f"{prefix}_grad_mean"] = obtener_magnitud_media_gradiente(path)
    f[f"{prefix}_grad_std"] = obtener_desv_estandar_gradiente(path)

    # textura
    f[f"{prefix}_glcm_contrast"] = obtener_contraste_textura(path)
    f[f"{prefix}_glcm_hom"] = obtener_homogeneidad_textura(path)
    f[f"{prefix}_glcm_energy"] = obtener_energia_textura(path)
    f[f"{prefix}_glcm_corr"] = obtener_correlacion_glcm(path)

    # color
    f[f"{prefix}_sat_mean"] = obtener_media_saturacion(path)
    f[f"{prefix}_sat_var"] = obtener_varianza_saturacion(path)

    return f

In [7]:
def extraer_features(path, prefix):
    f = {}

    f.update(features_generales(path, prefix))
    f.update(features_histograma(path, prefix))

    return f

In [8]:
import joblib
import pandas as pd
import numpy as np


FEATURES_FINALES = [
    'calidad',
    'orig_rango',
    'orig_std',
    'orig_michelson',
    'orig_hlfr',
    'orig_edges_s1',
    'orig_edges_s3',
    'orig_grad_std',
    'orig_glcm_hom',
    'orig_glcm_energy',
    'orig_glcm_corr',
    'orig_sat_mean',
    'orig_sat_var',
    'orig_hist_min',
    'orig_hf_log',
    'orig_lf_log',
    'orig_glcm_contrast_log',
    'metodo_ffmpeg',
    'metodo_imagemagick',
    'metodo_pillow',
    'formato_jpg',
    'formato_png',
    'formato_webp'
]


def extraer_features_modelo(
    img_path,
    metodo,
    formato,
    calidad
):
    
    # -------------------
    # Features originales
    # -------------------

    feats = extraer_features(
        img_path,
        "orig"
    )

    fila = {}

    fila["calidad"] = calidad

    fila["orig_rango"] = feats["orig_rango"]
    fila["orig_std"] = feats["orig_std"]
    fila["orig_michelson"] = feats["orig_michelson"]
    fila["orig_hlfr"] = feats["orig_hlfr"]

    fila["orig_edges_s1"] = feats["orig_edges_s1"]
    fila["orig_edges_s3"] = feats["orig_edges_s3"]

    fila["orig_grad_std"] = feats["orig_grad_std"]

    fila["orig_glcm_hom"] = feats["orig_glcm_hom"]
    fila["orig_glcm_energy"] = feats["orig_glcm_energy"]
    fila["orig_glcm_corr"] = feats["orig_glcm_corr"]

    fila["orig_sat_mean"] = feats["orig_sat_mean"]
    fila["orig_sat_var"] = feats["orig_sat_var"]

    fila["orig_hist_min"] = feats["orig_hist_min"]

    # -------------------
    # Transformaciones LOG
    # -------------------

    fila["orig_hf_log"] = np.log1p(
        feats["orig_hf"]
    )

    fila["orig_lf_log"] = np.log1p(
        feats["orig_lf"]
    )

    fila["orig_glcm_contrast_log"] = np.log1p(
        feats["orig_glcm_contrast"]
    )

    # -------------------
    # One Hot Metodo
    # -------------------

    fila["metodo_ffmpeg"] = int(
        metodo == "ffmpeg"
    )

    fila["metodo_imagemagick"] = int(
        metodo == "imagemagick"
    )

    fila["metodo_pillow"] = int(
        metodo == "pillow"
    )

    # -------------------
    # One Hot Formato
    # -------------------

    fila["formato_jpg"] = int(
        formato == "jpg"
    )

    fila["formato_png"] = int(
        formato == "png"
    )

    fila["formato_webp"] = int(
        formato == "webp"
    )

    return pd.DataFrame([fila])

In [16]:
def predecir_calidad(
    img_path,
    metodo,
    formato,
    calidad,
    modelo_path="modelos/xgb_final.pkl",
    scaler_path="modelos/scaler.pkl"
):

    modelo = joblib.load(modelo_path)
    scaler = joblib.load(scaler_path)

    X = extraer_features_modelo(
        img_path,
        metodo,
        formato,
        calidad
    )

    X = X[FEATURES_FINALES]

    cols_escalar = [
        c for c in X.columns
        if c.startswith("orig_")
    ]

    X[cols_escalar] = scaler.transform(
        X[cols_escalar]
    )

    pred = modelo.predict(X)[0]

    return float(pred)

In [20]:
score = predecir_calidad(
    img_path="prueba_img/image.png",
    metodo="ffmpeg",
    formato="jpg",
    calidad=1
)

print(score)

0.9670342206954956
